# Task 5 — Mini Quantum Toolkit
Toolkit untuk operasi aljabar linier dalam komputasi kuantum beserta verifikasi dari Task 2, 3, dan 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ket(*amps):
    """Membuat vektor kolom kompleks."""
    return np.array(amps, dtype=complex).reshape(-1, 1)

def dag(M):
    """Mengembalikan conjugate transpose."""
    return np.conjugate(M).T

def inner(a, b):
    """Menghitung inner product <a|b>."""
    return (dag(a) @ b).item()

def normalize(psi):
    """Mengembalikan state ternormalisasi."""
    norm = np.linalg.norm(psi)
    if norm == 0:
        raise ValueError("State nol tidak bisa dinormalisasi")
    return psi / norm

def is_hermitian(M):
    """Cek apakah M Hermitian."""
    return np.allclose(dag(M), M)

def is_unitary(M):
    """Cek apakah M Unitary."""
    I = np.eye(M.shape[0])
    return np.allclose(dag(M) @ M, I)

def probabilities(psi, basis):
    """Menghitung probabilitas pengukuran."""
    return [np.abs(inner(b, psi))**2 for b in basis]

def measure(psi, basis, shots):
    """Simulasi pengukuran state pada basis tertentu."""
    probs = probabilities(psi, basis)
    indices = np.random.choice(len(basis), p=probs, size=shots)
    counts = {i: 0 for i in range(len(basis))}
    for idx in indices:
        counts[idx] += 1
    return counts

In [ ]:
assert ket(1, 0).shape == (2, 1)
assert np.allclose(dag(np.array([[1, 1j], [-1j, 2]])), np.array([[1, 1j], [-1j, 2]]))
assert np.isclose(inner(ket(1, 0), ket(0, 1)), 0)
assert np.isclose(np.linalg.norm(normalize(ket(1, 1))), 1)
assert is_hermitian(np.array([[1, 0], [0, 1]]))
assert is_unitary(np.array([[0, 1], [1, 0]]))
assert np.allclose(probabilities(normalize(ket(1, 1)), [ket(1, 0), ket(0, 1)]), [0.5, 0.5])
assert sum(measure(normalize(ket(1, 1)), [ket(1, 0), ket(0, 1)], 100).values()) == 100

In [ ]:
ket0 = ket(1, 0)
ket1 = ket(0, 1)
ket_plus = normalize(ket(1, 1))
ket_minus = normalize(ket(1, -1))
comp_basis = [ket0, ket1]
had_basis = [ket_plus, ket_minus]

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)

### 1. Memverifikasi jawaban Task 2, 3, dan 4

In [ ]:
a = ket(1, 2j)
b = ket(2, -1j)
c = 0.5 * ket0 + (np.sqrt(3)/2 * np.exp(1j * np.pi/4)) * ket1

print("Verifikasi Task 2:")
print("<a|b> =", inner(a, b))
print("Norm a =", np.linalg.norm(a))
print("Prob c (Comp) =", probabilities(c, comp_basis))
print("Prob c (Had) =", probabilities(c, had_basis))

In [ ]:
A = np.array([[2, 1-1j], [1+1j, 3]], dtype=complex)
B = np.array([[1, 1], [0, 1]], dtype=complex)

print("Verifikasi Task 3:")
print("A hermitian?", is_hermitian(A))
print("S unitary?", is_unitary(S))
print("B unitary?", is_unitary(B))
print("HZH == X?", np.allclose(H @ Z @ H, X))
print("HXH == Z?", np.allclose(H @ X @ H, Z))
print("[X,Y] == 2iZ?", np.allclose(X @ Y - Y @ X, 2j * Z))

In [ ]:
evals_Y, _ = np.linalg.eig(Y)
evals_A, _ = np.linalg.eig(A)

print("Verifikasi Task 4:")
print("Eigenvalues Y =", evals_Y)
print("Eigenvalues A =", evals_A)

### 2. Simulasi State |0> dengan Hadamard

In [ ]:
state_2 = H @ ket0
c_comp_2 = measure(state_2, comp_basis, 1000)
c_had_2 = measure(state_2, had_basis, 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar(['|0>', '|1>'], [c_comp_2[0], c_comp_2[1]])
ax1.set_title("H|0> di Computational Basis")
ax2.bar(['|+>', '|->'], [c_had_2[0], c_had_2[1]])
ax2.set_title("H|0> di Hadamard Basis")
plt.show()

print("\nPenjelasan:")
print("H|0> menghasilkan state |+>. Jika diukur di computational basis, posisinya berupa superposisi rata 50:50. Tapi jika diukur di Hadamard basis, state selaras dengan basisnya sendiri (|+>), makanya terukur 100% di sana.")

### 3. Simulasi State SH|0>

In [ ]:
state_3 = S @ H @ ket0
c_comp_3 = measure(state_3, comp_basis, 1000)
c_had_3 = measure(state_3, had_basis, 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar(['|0>', '|1>'], [c_comp_3[0], c_comp_3[1]])
ax1.set_title("SH|0> di Computational Basis")
ax2.bar(['|+>', '|->'], [c_had_3[0], c_had_3[1]])
ax2.set_title("SH|0> di Hadamard Basis")
plt.show()

print("\nPenjelasan:")
print("Distribusi probabilitas di computational basis tetap sama (sekitar 50:50). Gate S hanya memberikan phase shift imajiner (i). Karena probabilitas dihitung dari nilai mutlak kuadrat amplitudonya, fase (i) ini tidak merubah hasil probabilitas akhirnya.")

### 4. State 2-Qubit |+> ⊗ |0>

In [ ]:
state_2q = np.kron(ket_plus, ket0)
basis_2q = [
    np.kron(ket0, ket0),
    np.kron(ket0, ket1),
    np.kron(ket1, ket0),
    np.kron(ket1, ket1)
]
probs_2q = probabilities(state_2q, basis_2q)

print("Probabilitas 2-Qubit |+> ⊗ |0>:")
print(f"P(|00>) = {probs_2q[0]:.4f}")
print(f"P(|01>) = {probs_2q[1]:.4f}")
print(f"P(|10>) = {probs_2q[2]:.4f}")
print(f"P(|11>) = {probs_2q[3]:.4f}")